# Human-in-the-Loop

## Review

We built an agent with memory that can:
* Use tools in a ReAct loop
* Persist state across turns via checkpointing

## Goals

In production, agents often need **human approval** before executing critical actions. LangGraph provides built-in support for this via **interrupts**.

We'll learn:
1. `interrupt_before` — pause the graph before a node executes
2. `interrupt_after` — pause after a node, before continuing
3. Reviewing pending actions and approving/rejecting them
4. Resuming execution after human input

![Human-in-the-Loop](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbb0c085deffbef02f9184_human-in-the-loop1.png)

In [ ]:
%run ../langchain_common.py

## Setup: Agent with Tools

Let's create a simple agent that can perform actions we might want to gate behind human approval.

In [ ]:
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to the specified recipient."""
    return f"Email sent to {to} with subject '{subject}'"

@tool
def search_contacts(query: str) -> str:
    """Search for a contact by name."""
    contacts = {"alice": "alice@company.com", "bob": "bob@company.com"}
    for name, email in contacts.items():
        if query.lower() in name:
            return f"Found: {name} ({email})"
    return f"No contact found for '{query}'"

tools = [send_email, search_contacts]
llm_with_tools = llm_noreason.bind_tools(tools)

In [ ]:
sys_msg = SystemMessage(content="You are a helpful assistant that can search contacts and send emails.")

def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

print("Graph defined (not yet compiled)")

## Interrupt Before: Gate Tool Execution

By passing `interrupt_before=["tools"]` to `compile()`, the graph will **pause** before executing any tool call.

This lets us inspect what the agent wants to do and decide whether to approve it.

In [ ]:
memory = MemorySaver()
graph = builder.compile(checkpointer=memory, interrupt_before=["tools"])

from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

## Run the Agent — It Will Pause

When the LLM decides to call a tool, the graph stops and waits for human approval.

In [ ]:
config = {"configurable": {"thread_id": "hitl-demo-1"}}

# Ask the agent to send an email
result = graph.invoke(
    {"messages": [HumanMessage(content="Send an email to Alice saying the meeting is at 3pm")]},
    config
)

print("Graph paused! Let's inspect what it wants to do:")
print()
last_msg = result["messages"][-1]
print(f"Agent wants to call: {last_msg.tool_calls}")

## Inspect the Pending State

We can use `get_state()` to see exactly where the graph is paused and what's pending.

In [ ]:
state = graph.get_state(config)
print(f"Next node to execute: {state.next}")
print(f"Number of messages so far: {len(state.values['messages'])}")
print()
print("Pending tool calls:")
for tc in state.values['messages'][-1].tool_calls:
    print(f"  Tool: {tc['name']}")
    print(f"  Args: {tc['args']}")

## Approve: Resume Execution

To approve, simply invoke the graph again with `None` input — it continues from where it paused.

In [ ]:
# Approve — resume execution
result = graph.invoke(None, config)

print("Execution resumed and completed!")
print()
for msg in result["messages"]:
    msg.pretty_print()

## Reject: Modify the State

To reject a tool call, we can update the state — replacing the AI's tool call message with a human override.

In [ ]:
# Start a new conversation
config2 = {"configurable": {"thread_id": "hitl-demo-2"}}

result = graph.invoke(
    {"messages": [HumanMessage(content="Send an email to Bob saying he's fired")]},
    config2
)

print("Agent wants to send a problematic email!")
print(f"Tool call: {result['messages'][-1].tool_calls}")

In [ ]:
from langchain_core.messages import ToolMessage

# Reject by providing a fake tool response that says "denied"
tool_call_id = result['messages'][-1].tool_calls[0]['id']

# Update state with a denial message
graph.update_state(
    config2,
    {"messages": [ToolMessage(content="Action denied by human reviewer. The email was not sent. Tell the user their request was rejected.", tool_call_id=tool_call_id)]},
    as_node="tools"  # Act as if the tools node produced this
)

# Resume — the agent will see the denial and respond accordingly
result = graph.invoke(None, config2)

print("After rejection:")
for msg in result["messages"]:
    msg.pretty_print()

## Key Takeaways

| Pattern | Use Case |
|---------|----------|
| `interrupt_before=["node"]` | Pause before a node executes (review before action) |
| `interrupt_after=["node"]` | Pause after a node (review results before continuing) |
| `graph.invoke(None, config)` | Resume/approve execution |
| `graph.update_state(config, ...)` | Modify state to reject/override |
| `graph.get_state(config)` | Inspect current pause point |

This pattern is **essential** for production agents where actions have real-world consequences (sending emails, making purchases, modifying databases).